# MotionJSON Colab Local UI Demo

This notebook launches the MotionJSON **Local UI** from a Colab runtime so you can use the browser workflow instead of only running API or CLI commands.

Recommended use: short, interactive CPU/no-model demos with explicit debug mock, threshold, motion foreground, or external masks. For real provider setup, hosted keys, or optional local SAM2/SAM3 cells, use `colab_ui_provider_connect_demo.ipynb`.

Colab GPU availability, memory, runtime length, and VM lifetime are not guaranteed. This notebook does not make SAM2/SAM3 runnable and should not be used for long-running/public UI hosting. Do not paste private videos, provider credentials, API keys, SAM checkpoints, or hosted-service secrets into a shared notebook.


## 1. Clone and install MotionJSON

This cell reuses an existing checkout when one already exists in the runtime. The `.[ui]` extra is intentionally lightweight in the current repo; the safe first path does not require SAM2, CUDA, detectors, model weights, or cloud credentials.


In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys
import textwrap
import time
import urllib.error
import urllib.request

REPO_URL = "https://github.com/ptse8204/json-animated-video.git"
REPO_DIR = Path("/content/json-animated-video") if Path("/content").exists() else Path("json-animated-video")

def run(cmd, *, cwd=None, check=True, capture=False):
    """Run a command with readable echoing for Colab and local notebooks."""
    if isinstance(cmd, str):
        display_cmd = cmd
        shell = True
    else:
        display_cmd = " ".join(shlex.quote(str(part)) for part in cmd)
        shell = False
    print(f"$ {display_cmd}")
    completed = subprocess.run(
        cmd,
        cwd=cwd,
        check=check,
        shell=shell,
        text=True,
        capture_output=capture,
    )
    if capture:
        if completed.stdout:
            print(completed.stdout)
        if completed.stderr:
            print(completed.stderr, file=sys.stderr)
    return completed

if not REPO_DIR.exists():
    run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    print(f"Using existing checkout: {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Repository: {Path.cwd()}")
run([sys.executable, "-m", "pip", "install", "-U", "pip"])
run([sys.executable, "-m", "pip", "install", "-e", ".[ui]"])


## 2. Create a deterministic sample video

The UI can register any readable video path inside the Colab runtime. This creates the built-in red-ball clip so the UI has a known local file to use.


In [ ]:
DEMO_VIDEO = REPO_DIR / "examples" / "demo_red_ball.mp4"
run([sys.executable, "examples/make_demo_video.py", "--out", str(DEMO_VIDEO)])
print(f"Use this source video path in the UI: {DEMO_VIDEO}")


## 3. Check provider diagnostics

Diagnostics keep optional providers honest. The no-model UI path should work even when SAM2, CUDA, hosted segmentation, detector models, or FFmpeg are unavailable.


In [ ]:
run([sys.executable, "-m", "motionjson.cli", "backend", "diagnostics", "--text"])


## 4. Launch the local UI server

The server binds to `127.0.0.1` inside the notebook VM. The next cell uses Colab's notebook port proxy to display `/ui/` as an iframe or a window.


In [ ]:
UI_PORT = 8766
UI_ROOT = REPO_DIR / ".motionjson" / "colab-ui"
UI_ROOT.mkdir(parents=True, exist_ok=True)
DB_PATH = UI_ROOT / "backend.sqlite"
STORAGE_ROOT = UI_ROOT / "storage"
LOG_PATH = UI_ROOT / "ui.log"

def wait_for_url(url: str, *, timeout_seconds: int = 45) -> None:
    deadline = time.time() + timeout_seconds
    last_error = None
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=2) as response:
                if response.status < 500:
                    return
        except Exception as exc:  # noqa: BLE001 - printed for notebook users.
            last_error = exc
        time.sleep(1)
    log_tail = ""
    if LOG_PATH.exists():
        log_tail = "\n" + "".join(LOG_PATH.read_text(errors="replace").splitlines(True)[-40:])
    raise RuntimeError(f"UI did not become ready at {url}. Last error: {last_error}{log_tail}")

# Stop an older server from a previous run of this cell.
try:
    ui_process.terminate()  # type: ignore[name-defined]
    ui_process.wait(timeout=5)  # type: ignore[name-defined]
except Exception:
    pass

ui_command = [
    sys.executable,
    "-m",
    "motionjson.cli",
    "ui",
    "--no-open",
    "--debug-mock",
    "--host",
    "127.0.0.1",
    "--port",
    str(UI_PORT),
    "--db",
    str(DB_PATH),
    "--storage-root",
    str(STORAGE_ROOT),
]
print("Starting MotionJSON UI:")
print(" ".join(shlex.quote(part) for part in ui_command))
log_file = LOG_PATH.open("w", encoding="utf-8")
ui_process = subprocess.Popen(
    ui_command,
    cwd=REPO_DIR,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
)
wait_for_url(f"http://127.0.0.1:{UI_PORT}/api/health")
print(f"MotionJSON UI is running on port {UI_PORT}.")
print(f"Log file: {LOG_PATH}")
print(f"Demo video path to register in the UI: {DEMO_VIDEO}")


## 5. Open the UI

In the UI:

1. Use `Start` to choose the task. For this notebook, prefer `Find moving things`, `Import masks`, or the debug no-model smoke path.
2. Use `Video` to register this source video path: `/content/json-animated-video/examples/demo_red_ball.mp4`.
3. Use `Model` only when the chosen task requires one; this notebook is meant for CPU/no-model work.
4. Use `Prepare & run` to validate the readable run plan and start the run.
5. Watch **Job Center** / **Run monitor** after the run starts.
6. Use `Review & export` to follow `Candidates` -> `Track selected` -> `Tracks` -> `Corrections` -> `Export`.

For a larger screen, use the window link printed by the cell after the iframe.


In [ ]:
try:
    from google.colab import output  # type: ignore

    output.serve_kernel_port_as_iframe(UI_PORT, path="/ui/", height=900)
    print("Open the same UI in a separate browser tab/window:")
    output.serve_kernel_port_as_window(UI_PORT, path="/ui/")
except Exception as exc:  # Works outside Colab as a normal local URL.
    print(f"Colab port proxy is not available: {exc}")
    print(f"Open locally: http://127.0.0.1:{UI_PORT}/ui/")


## 6. Optional: inspect the UI server log

The UI workspace can contain local database rows and user artifacts, so this notebook does not zip or download it. Use this log tail for lightweight troubleshooting without packaging provider settings or generated files.


In [ ]:
if LOG_PATH.exists():
    lines = LOG_PATH.read_text(errors="replace").splitlines()
    print("\n".join(lines[-80:]) or "Log file is empty.")
else:
    print(f"No UI log file yet at {LOG_PATH}")


## 7. Stop the UI server when you are done

Stopping the server frees the port and makes reruns cleaner.


In [ ]:
try:
    ui_process.terminate()
    ui_process.wait(timeout=10)
    print("Stopped MotionJSON UI server.")
except Exception as exc:
    print(f"UI process was already stopped or unavailable: {exc}")
